# ITBS-VoGen — Colab Training

Google Colab (T4 GPU) で RVC 話者モデルを学習するためのノートブック。ローカル (Mac) では推論のみ行い、重い学習だけクラウドに逃がす運用。

## 事前準備

1. **Google Drive に学習データをアップロード**: `MyDrive/itbs-vogen/Train/` 以下に `set1/*.wav` を配置する。
2. **ランタイム変更**: メニュー → ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータ = **T4 GPU**。
3. 以下のセルを上から順に実行。

## 1. GPU 確認

In [ ]:
!nvidia-smi

## 2. Google Drive マウント

学習データ読み込みと成果物保存に使用。認証プロンプトが出るので承認する。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. パス設定

必要なら編集する。`DRIVE_ROOT` 配下の構造:
```
itbs-vogen/
├── Train/set1/*.wav   ← 事前にアップロード
└── models/            ← 学習後の model.pth / model.index をここに吐く
```

In [ ]:
import os
from pathlib import Path

# === ユーザー設定 ===
SPEAKER_NAME = 'itbs_singer'       # 学習するモデル名
TOTAL_EPOCHS = 200                 # 典型値: 150-300
SAVE_EVERY_EPOCH = 50              # 途中保存の粒度
BATCH_SIZE = 12                    # T4 なら 12、A100 なら 24+
SR = '48k'                         # 32k / 40k / 48k
VERSION = 'v2'

# === Drive 上のパス ===
DRIVE_ROOT = Path('/content/drive/MyDrive/itbs-vogen')
DRIVE_TRAIN_DIR = DRIVE_ROOT / 'Train' / 'set1'
DRIVE_OUTPUT_DIR = DRIVE_ROOT / 'models' / SPEAKER_NAME

# === Colab 作業ディレクトリ ===
WORK_DIR = Path('/content/ITBS-VoGen')

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert DRIVE_TRAIN_DIR.exists(), f'Training data not found: {DRIVE_TRAIN_DIR}'
train_wavs = list(DRIVE_TRAIN_DIR.glob('*.wav'))
print(f'Training files: {len(train_wavs)}')
for w in train_wavs[:5]:
    print(f'  {w.name}')

## 4. リポジトリ取得

公開リポジトリとRVC submoduleを取得。

In [ ]:
%cd /content
![ -d /content/ITBS-VoGen ] || git clone --recurse-submodules https://github.com/Arimuri/ITBS-VoGen.git
%cd /content/ITBS-VoGen
!git pull
!git submodule update --init --recursive

## 5. 依存インストール

Colab は Linux x86_64 なので RVC の requirements.txt をそのまま使える (Mac で問題だった aria2 も通る)。5-10分程度。

In [ ]:
%cd /content/ITBS-VoGen
# RVC depends on an old omegaconf whose metadata trips pip >= 24.1.
!pip install -q 'pip<24.1'
!pip install -q -r third_party/rvc/requirements.txt
!pip install -q -e .


## 6. ベースモデル取得

HuBERT + RMVPE + pretrained_v2 (48k, F0あり) をDL。初回のみ。

In [ ]:
!bash scripts/download_models.sh

## 7. 学習データ配置

Drive のデータを Colab ローカル (`/content/ITBS-VoGen/Train/set1/`) にコピーする (Drive直読みは遅い)。

In [ ]:
local_train = WORK_DIR / 'Train' / 'set1'
local_train.mkdir(parents=True, exist_ok=True)
!cp -v {DRIVE_TRAIN_DIR}/*.wav {local_train}/
!ls -la {local_train}

## 8. 学習実行

T4 で 29分データ × 200 epoch ≈ **約20-40分**。

途中で `save_every_epoch` ごとに `assets/weights/<name>_e<epoch>_s<step>.pth` が保存され、最終エポック後に `models/speakers/<name>/` に `model.pth` + `model.index` が書き出される。

In [ ]:
!itbs-vogen train \
    -s {SPEAKER_NAME} \
    -d {local_train} \
    --sr {SR} \
    --version {VERSION} \
    --epochs {TOTAL_EPOCHS} \
    --save-every {SAVE_EVERY_EPOCH} \
    --batch-size {BATCH_SIZE} \
    --n-proc 4 \
    --device cuda:0

## 9. 成果物を Drive に保存

学習後の `model.pth` + `model.index` を Google Drive にコピーする。これをローカルMacにDLして推論に使う。

In [ ]:
import shutil

local_model = WORK_DIR / 'models' / 'speakers' / SPEAKER_NAME
assert (local_model / 'model.pth').exists(), 'Training output not found'

shutil.copy(local_model / 'model.pth',   DRIVE_OUTPUT_DIR / 'model.pth')
shutil.copy(local_model / 'model.index', DRIVE_OUTPUT_DIR / 'model.index')

print(f'Saved to Drive: {DRIVE_OUTPUT_DIR}')
!ls -la {DRIVE_OUTPUT_DIR}

## 10. ローカル (Mac) で使う

Mac 側で以下を実行:

1. Google Drive から `model.pth` + `model.index` をダウンロード
2. ローカルリポジトリの `models/speakers/<SPEAKER_NAME>/` に配置
3. 推論:
   ```bash
   itbs-vogen infer inputs/test_HonestyVo.wav \
       -o outputs/test_HonestyVo_restored.wav \
       -s itbs_singer
   ```

## トラブルシュート

- **OOM (CUDA out of memory)**: `BATCH_SIZE` を 8, 6, 4 と下げる。
- **学習が途中で止まる**: ランタイムが切れた可能性。再接続して同じセルを再実行すると、`assets/weights/` の途中保存から`-l 1`で継続できる (現状は再スタートになる仕様)。
- **品質が低い**: epoch数を増やす (300-500)、または学習データを追加して再学習。